In [18]:
import pandas as pd

#zip file processing
import zipfile

#directory controls
import os

import io

In [19]:
#change to your file directory
ERC_DATA = "..\\datasets\\ERC20-stablecoins.zip"
GFC_DATA="..\\datasets\\gfc.zip"

In [20]:
os.makedirs("../datasets/processed/erc_20", exist_ok=True)  # creates nested directories
os.makedirs("../datasets/processed/gfc_data", exist_ok=True)

In [21]:
def describe(f, name):
    """
    To understand more about the dataframe (columns, num of null values, shape)
    Args:
        f -> String representing : file path
        name -> String : file name
    Output:
        None
    """
    if name.startswith("event_data"):
        df = pd.read_csv(f, encoding='latin-1')
    else:
        df = pd.read_csv(f)
    print(f"\n📋 All columns in {name}")
    print(df.columns.tolist())
    print(f"\n📊 First 5 rows:")
    print(df.head(5))
    print(f"\n Dataframe shape")
    print(df.shape)
    print(f"\n Number of null values")
    print(df.isna().sum())
    return df

# Processing ERC 20 Data

In [22]:
data_dataframes = {}
with zipfile.ZipFile(ERC_DATA) as z:
    for name in z.namelist():

        if name.endswith(".zip"):
            # if it is another zipped folder
            print(f"Opening nested zip {name}")
            nested_zip = z.read(name)

            with zipfile.ZipFile(io.BytesIO(nested_zip)) as nested_z:
                for fileName in nested_z.namelist():
                    with nested_z.open(fileName) as nested_f:
                        df = pd.read_csv(nested_f)
                        df['date'] = pd.to_datetime(df["timestamp"], unit="s") #convert unix timestamp to date time
                        df["coins"] = fileName.split("_")[0] #get the coin name as column
                        df.to_csv(f"../datasets/processed/erc_20/processed_{fileName.split("/")[1]}", index=False)
        else:
            if ("token" in name):
                continue
        
            else:
                print(f"Opening file {name}")
                # if it is a file
                with z.open(name) as f:
                    df = describe(f, name)

                    if ("timestamp" in df.columns): #conditions according to column name
                        df["date"] = pd.to_datetime(df["timestamp"], unit="s") #convert unix timestamp to date time
                    elif ("time_stamp" in df.columns):
                        df["date"] = pd.to_datetime(df["time_stamp"], unit="s") #convert unix timestamp to date time
                    df.to_csv(f"../datasets/processed/erc_20/processed_{name}", index=False)

Opening nested zip price_data.zip
Opening file event_data.csv

📋 All columns in event_data.csv
['event', 'timestamp', 'type', 'stablecoin']

📊 First 5 rows:
                                               event   timestamp      type  \
0  BlackRock and Fidelity Back USDC in $400 Milli...  1649721600  positive   
1  Terra UST takes over BUSD to become third larg...  1650412800  positive   
2  LARGE amounts of UST selling on ANCHOR (approx...  1651881600  negative   
3  UST depegs LFG deploys assets to defend peg (7...  1651968000  negative   
4    UST Depegs again to 35 cents LUNA keeps falling  1652054400  negative   

  stablecoin  
0       usdc  
1       ustc  
2       ustc  
3       ustc  
4       ustc  

 Dataframe shape
(38, 4)

 Number of null values
event         0
timestamp     0
type          0
stablecoin    0
dtype: int64


# Processing GFC Data

In [23]:
# for filename in os.listdir(GFC_DATA_FOLDER):
#     if filename.endswith(".csv"):
#         file_path = os.path.join(GFC_DATA_FOLDER, filename)
#         if os.path.isfile(file_path):
#             print(filename)

#             df = describe(file_path, filename) #understand the file
#             print(f"Processing file for {df.iloc[0,1]}")
#             #formating df's format
#             ticker = df.iloc[0, 1] #get ticker value
#             date= df.iloc[2:, 0] # get date column
#             df = df.iloc[3:,:] # get the relevant dataset (from row 2 onwards)
#             df['ticker'] = ticker #assigned ticker to ticker column
#             df['date'] = date #assigned date to date column
#             df.to_csv(f"../datasets/processed/gfc_data/processed_{filename}", index=False)


In [24]:
def process_gfc_data(source_path, output_folder):
    # Ensure output directory exists
    os.makedirs(output_folder, exist_ok=True)

    # CASE 1: Source is a ZIP File
    if zipfile.is_zipfile(source_path):
        print(f"📦 Processing ZIP file: {source_path}")
        with zipfile.ZipFile(source_path, "r") as z:
            for filename in z.namelist():
                if filename.endswith(".csv"):
                    with z.open(filename) as f:
                        process_and_save(f, filename, output_folder)

    # CASE 2: Source is a Directory (Folder)
    elif os.path.isdir(source_path):
        print(f"📂 Processing Directory: {source_path}")
        for root, dirs, files in os.walk(source_path):
            for filename in files:
                if filename.endswith(".csv"):
                    file_path = os.path.join(root, filename)
                    with open(file_path, "rb") as f:
                        process_and_save(f, filename, output_folder)
    else:
        print("❌ Invalid source path provided.")

def process_and_save(file_handle, filename, output_folder):
    # Your group mate's cleaning logic
    df = pd.read_csv(file_handle)
    
    # Extract the clean filename (removing any path prefix)
    clean_name = os.path.basename(filename)
    
    # Data formatting logic
    ticker = df.iloc[0, 1] 
    date_col = df.iloc[2:, 0] 
    processed_df = df.iloc[3:, :].copy() 
    processed_df['ticker'] = ticker
    processed_df['date'] = date_col
    
    # Save the processed CSV
    save_path = os.path.join(output_folder, f"processed_{clean_name}")
    processed_df.to_csv(save_path, index=False)
    print(f"✅ Saved: {save_path}")

# Example Usage:
process_gfc_data(GFC_DATA, "../datasets/processed/gfc_data/")


📦 Processing ZIP file: ..\datasets\gfc.zip
✅ Saved: ../datasets/processed/gfc_data/processed_AIG.csv
✅ Saved: ../datasets/processed/gfc_data/processed_^VIX.csv
✅ Saved: ../datasets/processed/gfc_data/processed_C.csv
✅ Saved: ../datasets/processed/gfc_data/processed_JPM.csv
✅ Saved: ../datasets/processed/gfc_data/processed_^GSPC.csv
✅ Saved: ../datasets/processed/gfc_data/processed_WGS3MO.csv
✅ Saved: ../datasets/processed/gfc_data/processed_^DJI.csv
✅ Saved: ../datasets/processed/gfc_data/processed_TEDRATE.csv
